# Bilderkennung mit CNNs

Jetzt wollen wir für den Gestensatz einen Classifier mit State of the Art Performance trainieren. Dafür laden wir zuerst den Datensatz wie zuvor ein. CNNs können mit weniger Parameter gut mit größeren Bildern umgehen, daher starten wir direkt mit einer Größe von 96x96.

In [1]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory

BATCH_SIZE = 64
IMG_SIZE = (96, 96)
directory = "../UB6/Dataset/"
train_dataset = image_dataset_from_directory(directory=directory,
                                             shuffle=True,
                                             batch_size=BATCH_SIZE,
                                             image_size=IMG_SIZE,
                                             validation_split=0.2,
                                             subset='training',
                                             seed=42)
validation_dataset = image_dataset_from_directory(directory=directory,
                                             shuffle=True,
                                             batch_size=BATCH_SIZE,
                                             image_size=IMG_SIZE,
                                             validation_split=0.2,
                                             subset='validation',
                                             seed=42)

Found 2071 files belonging to 10 classes.
Using 1657 files for training.
Found 2071 files belonging to 10 classes.
Using 414 files for validation.


## Das CNN

Jetzt wollen wir ein Convolutional Neural Network erstellen. Das funktioniert sehr ähnlich, wie zuvor, nur dass wir zwischen "Rescaling" und "Flatten" noch ein paar Layer einfügen. Zum Beispiel 3 mal je eine 2D Convolution gefolgt von einem Max Pooling Layer:
- `tf.keras.layers.Conv2D`:
    - `filters`: Wie viele Output Channel wollen wir haben (tendenziell mindestens soviele wie reinkamen)
    - `kernel_size`: Wieviele Pixel die Convolution betrifft (meistens so zwischen 3 und 11)
    - `activation`: Die Activation Function (analog zu anderen Layern)
    - `padding`: In der Regel entweder "valid" oder "same". Mit "same" bleibt die Output Höhe und Breite gleich (oft ist das ein guter Default zum Starten).
- ` tf.keras.layers.MaxPool2D`:
    - `pool_size`: Wie viele Pixel in jeder Richtung zusammengefasst werden. Um diesen Faktor verkleinert sich das Bild.
    
Dieses neuronale Netz können wir genauso trainieren wie zuvor. Je tiefer das Netz wird, umso wackeliger wird Stochastic Gradient Descent. Eine Erweiterung davon ist der der Adam Optimizer (`tf.optimizers.Adam`). Dieser macht läuft auch den Gradienten entlang, ist aber etwas stabiler (indem er sich merkt, in welche Richtung er zuvor um wieviel gelaufen ist). Auch hier geben wir die Lernrate mit (und können eine mit der Zeit sinkende Lernrate angeben).

Mit `model.summary()` können Sie sich auch wieder ausgeben lassen, wie dein neuronales Netzwerk aussieht. Je nach Definition der Layer kommen wir hier mit deutlich weniger Parametern aus, als in der Übung zuvor. Trotzdem brauchen wir eventuell mehr Parameter, als wir Trainingsbeispiele haben.

Nach ein paar Runden experimentieren bin ich auf rund 92% Accuracy auf dem Validation Set gekommen. Damit sind wir schon einen guten Schluck besser, als die SVMs auf den gleichen Daten. Mit mehr Fine Tuning ist hier sicher auch noch mehr möglich.

In [34]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(96, 96, 3)),
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Conv2D(filters = 16, kernel_size= 5, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(filters = 32, kernel_size= 5, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(filters = 64, kernel_size= 5, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=10, #jedes neuron bekommt 784 features und 1 bias. Bias(b) ist intercept. allgemein für logistische regression: z=w1x1+w2x2+wnxn+b -> activation function -> output
                           activation= "softmax"
    )
])

# ADD CODE HERE

model.compile( 
    optimizer = tf.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics = [tf.metrics.SparseCategoricalAccuracy]
)
# ADD CODE HERE

model.summary()

Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_13 (Rescaling)        │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_39 (Conv2D)              │ (None, 92, 92, 16)     │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_39 (MaxPooling2D) │ (None, 46, 46, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_40 (Conv2D)              │ (None, 42, 42, 32)     │        12,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_40 (MaxPooling2D) │ (None, 21, 21, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_41 (Conv2D)              │ (None, 17, 17, 64)     │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_41 (MaxPooling2D) │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_13 (Flatten)            │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 10)             │        40,970 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 106,282 (415.16 KB)

 Trainable params: 106,282 (415.16 KB)

 Non-trainable params: 0 (0.00 B)

In [35]:
# ADD CODE HERE
epochs = 12

history = model.fit(
    train_dataset,
    validation_data = validation_dataset,
    epochs = epochs
)

Epoch 1/12
26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 136ms/step - loss: 2.2880 - sparse_categorical_accuracy: 0.1563 - val_loss: 2.2236 - val_sparse_categorical_accuracy: 0.2874
Epoch 2/12
26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 140ms/step - loss: 1.5833 - sparse_categorical_accuracy: 0.5250 - val_loss: 0.9870 - val_sparse_categorical_accuracy: 0.6256
Epoch 3/12
26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 143ms/step - loss: 0.7740 - sparse_categorical_accuracy: 0.7471 - val_loss: 0.7509 - val_sparse_categorical_accuracy: 0.7440
Epoch 4/12
26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 147ms/step - loss: 0.6053 - sparse_categorical_accuracy: 0.8063 - val_loss: 0.5671 - val_sparse_categorical_accuracy: 0.8068
Epoch 5/12
26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 144ms/step - loss: 0.4563 - sparse_categorical_accuracy: 0.8582 - val_loss: 0.5193 - val_sparse_categorical_accuracy: 0.8309
Epoch 6/12
26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 145ms/step - loss: 0.3558 - sparse_categorical_accuracy: 0.8992 - val_loss: 0.3904 - val_sparse_categorical_accuracy: 0.8720
Epoc

## Transfer Learning

Was machen wir, wenn wir weniger Daten zur Verfügung haben? Simulieren können wir das, indem wir die größe unseres Validation Sets auf 80% erhöhen:

In [ ]:
train_dataset = image_dataset_from_directory(directory=directory,
                                             shuffle=True,
                                             batch_size=BATCH_SIZE,
                                             image_size=IMG_SIZE,
                                             validation_split=0.8,
                                             subset='training',
                                             seed=42)
validation_dataset = image_dataset_from_directory(directory=directory,
                                             shuffle=True,
                                             batch_size=BATCH_SIZE,
                                             image_size=IMG_SIZE,
                                             validation_split=0.8,
                                             subset='validation',
                                             seed=42)

Wir wollen nun mit einem vortrainierten Neuronalen Netzwerk starten. `tf.keras.applications.EfficientNetB0` implementiert das kleinste EfficientNet und hat bereits auf "imagenet" vortrainierte Gewichte. Mit wir können uns die Struktur davon direkt anschauen:

In [2]:
tf.keras.applications.EfficientNetB0(include_top=True,
                                     weights='imagenet').summary()

21834768/21834768 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


Model: "efficientnetb0"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 5,330,571 (20.33 MB)

 Trainable params: 5,288,548 (20.17 MB)

 Non-trainable params: 42,023 (164.16 KB)

Dem Konstruktor von `EfficientNetB0` können wir mit `input_shape=` auch einen anderen Input Shape mitgeben. Meistens sind nur bestimmte Kombinationen möglich - das ist hier auch der Grund, weswegen wir unseren Input als 96x96 einlesen, was am nächsten am Original Input von 100x100 ist. Da wir als Output nicht die 1000 Klassen von ImageNet, sondern unsere eigenen 10 Klassen haben, sollten wir `include_top` ausschalten.

In [5]:
base_model = tf.keras.applications.EfficientNetB0(input_shape = (96, 96, 3), include_top = False)

base_model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "efficientnetb0"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 96, 96, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 96, 96, 3) │          0 │ input_layer_1[0]… │
│ (Rescaling)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_1     │ (None, 96, 96, 3) │          7 │ rescaling_2[0][0] │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_3         │ (None, 96, 96, 3) │          0 │ normalization_1[… │
│ (Rescaling)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 97, 97, 3) │          0 │ rescaling_3[0][0] │
│ (ZeroPadding2D)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 48, 48,    │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 48, 48,    │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 48, 48,    │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 48, 48,    │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 48, 48,    │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 48, 48,    │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 48, 48,    │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 48, 48,    │        512 │ block1a_se_excit

 Total params: 4,049,571 (15.45 MB)

 Trainable params: 4,007,548 (15.29 MB)

 Non-trainable params: 42,023 (164.16 KB)

Wenn wir die Summaries vergleichen, so sehen wir, dass ohne "top" 3 Layer am Ende fehlen. Für diese können wir nun unsere eigenen hinzufügen. Leider können wir nun nicht mehr "Sequential" nehmen, um das Modell zusammenzubauen, sondern müssen die etwas flexiblere "Functional API" von Tensorflow verwenden. Damit wird es aber auch nicht deutlich schwieriger.

In der Functional API definieren wir welche Inputs wir haben und wie die Outputs aus dem Input entstehen.
Hier ein Beispiel für 2 lineare Layer mit Functional API:

In [7]:
inputs = tf.keras.Input(shape= (96, 96, 3))
x = tf.keras.layers.Dense(20, activation="relu")(inputs)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 96, 96, 20)     │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 96, 96, 1)      │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101 (404.00 B)

 Trainable params: 101 (404.00 B)

 Non-trainable params: 0 (0.00 B)

Analog können wir nun unser Modell definieren.

- Als erstes muss der Input in `tf.keras.applications.efficientnet.preprocess_input` gegeben werden. Die Funktion sorgt dafür, dass die Bilder die Annahmen von EfficientNet erfüllen (zum Beispiel die Pixel zwischen 0 und 1 skaliert werden).
- Nun wenden wir unser `base_model` an.
- Jetzt kommen die 3 Layer, die wir ersetzen müssen:
 - `tf.keras.layers.GlobalAveragePooling2D` - Das ist eine Alternative zu "Flatten", die einfach das Maximum je Channel nimmt.
 - `tf.keras.layers.Dropout` (mit Parameter 0.2) - Dropout ist eine Regularisierungstechnik für neuronale Netze
 - `tf.keras.layers.Dense` - der Output, den wir produzieren wollen
 
Bevor wir nun das Modell trainieren müssen wir Tensorflow sagen, dass die Parameter von `base_model` nicht verändert werden sollen. Das tun wir, indem wir den Parameter `.trainable` von `base_model` auf `False` setzen.

Jetzt trainieren wir unser Modell analog wie zuvor. Da das Validation Set recht groß ist, wollen wir das vielleicht beim Training weg lassen, um das ganze zu beschleunigen. Wir können die Metriken am Ende separat mit `model.evaluate` berechnen.

In [17]:
inputs = tf.keras.Input(shape=(96,96,3))
x = tf.keras.applications.efficientnet.preprocess_input(inputs)
base_model = tf.keras.applications.EfficientNetB0(include_top = False)
x = base_model(x)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(10, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_12 (InputLayer)     │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 3, 3, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,062,381 (15.50 MB)

 Trainable params: 4,020,358 (15.34 MB)

 Non-trainable params: 42,023 (164.16 KB)

Trotz weniger Daten kommen wir so auf eine Accuracy, die mit unserem SVM Modell mithalten kann. Wir können als nächstes das Modell noch ein wenig Finetunen. Dafür trainieren wir noch ein paar Epochen, wobei wir auch noch ein paar Layer vom EfficientNet anpassen.

Erstmal schauen wir, wieviele Layer das EfficientNet base_model hat:

In [18]:
len(base_model.layers)

238

Nun wollen wir `.trainable` für das `base_model` auf `True` und nur für einzelne Layer auf `False` setzen. Frieren Sie so die Parameter für alle, außer die letzten 10 Layer ein.

Anschließend können Sie das Model noch ein paar (~10) Epochen weiter trainieren. Hierbei sollte die Lernrate deutlich niedriger sein. Je mehr Layer wir trainieren, umso größer ist das Risiko, dass wir nen großen Schritt in die falsche Richtung machen und umso vorsichtiger müssen wir sein. Außerdem haben wir recht wenige Daten und wollen nicht zu stark overfitten. Indem wir nur wenige Epochen trainieren sorgen wir dafür, dass wir im Zweifel nicht zu viel Schaden anrichten.

In [20]:
# ADD CODE HERE

base_model.trainable = True

count = 0

for layer in base_model.layers:
    
    if(count>231):
        break
    layer.trainable = False
    count +=1



In [27]:
model.compile(

    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),

    loss=tf.keras.losses.SparseCategoricalCrossentropy(),

    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()]

)

In [28]:
# ADD CODE HERE
epochs = 10

history = model.fit(
    train_dataset,
    validation_data = validation_dataset,
    epochs = epochs
)

Epoch 1/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 11s 267ms/step - loss: 0.2696 - sparse_categorical_accuracy: 0.9415 - val_loss: 0.3184 - val_sparse_categorical_accuracy: 0.9106
Epoch 2/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 6s 240ms/step - loss: 0.2746 - sparse_categorical_accuracy: 0.9342 - val_loss: 0.3160 - val_sparse_categorical_accuracy: 0.9106
Epoch 3/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 7s 285ms/step - loss: 0.2638 - sparse_categorical_accuracy: 0.9384 - val_loss: 0.3129 - val_sparse_categorical_accuracy: 0.9106
Epoch 4/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 7s 265ms/step - loss: 0.2616 - sparse_categorical_accuracy: 0.9378 - val_loss: 0.3105 - val_sparse_categorical_accuracy: 0.9106
Epoch 5/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 6s 249ms/step - loss: 0.2579 - sparse_categorical_accuracy: 0.9403 - val_loss: 0.3078 - val_sparse_categorical_accuracy: 0.9106
Epoch 6/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 7s 279ms/step - loss: 0.2513 - sparse_categorical_accuracy: 0.9505 - val_loss: 0.3057 - val_sparse_categorical_accuracy: 0.9106
Epo